In [1]:
import numpy as np

# -----------------------------
# 1. Parameter space
# -----------------------------
Ns = 2 ** np.arange(9)        # 2^0 to 2^8
vals = 2 ** np.arange(26)     # 2^0 to 2^25
# Ns = np.arange(1, 11)        # 1 to 10
# vals = np.arange(1, 11)     # 1 to 10

In [2]:

# All configurations (N, T, W)
N, T, W = np.meshgrid(Ns, vals, vals, indexing='ij')

N = N.ravel()
T = T.ravel()
W = W.ravel()

num = len(N)
print(f"Total configs: {num}")  # should be 1000

# -----------------------------
# 2. Pairwise preference matrix
# -----------------------------
Ni = N[:, None]
Nj = N[None, :]

Ti = T[:, None]
Tj = T[None, :]

Wi = W[:, None]
Wj = W[None, :]

lhs = Nj * Ti * (Wj + Tj + (Ti / 2.0))
rhs = Ni * Tj * (Wi + Ti + (Tj / 2.0))

P = lhs < rhs   # P[i, j] = True means i ≺ j

print("Pairwise matrix computed.")


Total configs: 6084
Pairwise matrix computed.


In [3]:

def get_rmse_sq(ntw_1, ntw_2):
    elapsed = 0
    total_score = 0
    for x in (ntw_1, ntw_2):
        N, T, W = x
        elapsed += T
        total_score += N * ((W + elapsed) ** 2)
    return total_score

idx_to_ntw = dict()


for i, N1 in enumerate(Ns):
    for j, T1 in enumerate(vals):
        for k, W1 in enumerate(vals):
            idx1 = i * len(vals) * len(vals) + j * len(vals) + k
            idx_to_ntw[idx1] = (N1, T1, W1)


# for i, N1 in enumerate(Ns):
#     for j, T1 in enumerate(vals):
#         for k, W1 in enumerate(vals):
#             for l, N2 in enumerate(Ns):
#                 for m, T2 in enumerate(vals):
#                     for n, W2 in enumerate(vals):
#                         idx1 = i * len(vals) * len(vals) + j * len(vals) + k
#                         idx2 = l * len(vals) * len(vals) + m * len(vals) + n
#                         idx_to_ntw[idx1] = (N1, T1, W1)
#                         if get_rmse_sq((N1, T1, W1), (N2, T2, W2)) < get_rmse_sq((N2, T2, W2), (N1, T1, W1)):
#                             assert P[idx1, idx2], P[idx1, idx2]
#                         else:
#                             assert not P[idx1, idx2], P[idx1, idx2]



In [4]:
from numba import njit, prange, types
import numpy as np
from numba.typed import List


@njit(fastmath=True, nogil=True, boundscheck=False, parallel=True)
def counterexample_frequency_parallel(P):
    num = len(P)
    
    # 1. Define the type: A 1D array of 64-bit integers
    list_item_type = types.int64[:]
    result = List.empty_list(list_item_type)
    
    # 2. Pre-fill with truly empty arrays (size 0)
    # This allocates the 'slots' in the list so result[index] works in parallel
    empty_arr = np.empty(0, dtype=np.int64)
    for _ in range(num * num):
        result.append(empty_arr)

    total_counterexamples = 0

    for i in prange(num):
        local_counterexamples = 0
        
        # Get indices where P[i] is True to iterate over j
        possible_j = np.where(P[i])[0]
        
        for j in possible_j:
            if j == i:
                continue

            # Compute mask: k where P[j,k] is True AND P[i,k] is False
            mask = P[j, :] & (~P[i, :])
            mask[i] = False
            mask[j] = False

            # Extract the actual indices
            indices = np.where(mask)[0]
            
            # Increment counter by the number of found indices
            local_counterexamples += indices.shape[0]
            
            # Place the found indices into the pre-allocated slot
            result[i * num + j] = indices

        total_counterexamples += local_counterexamples

    # Stats
    total_triples = num * (num - 1) * (num - 2)
    
    percentage = (total_counterexamples / total_triples) * 100
    
    print("Total triples:", total_triples)
    print("Counterexamples:", total_counterexamples)
    print("Counterexample percentage:", percentage)

    return result

def convert_to_triplets(results, num):
    # 1. Filter out the empty arrays from your results
    # We only care about entries where counterexamples were actually found
    valid_indices = [(idx, k_array) for idx, k_array in enumerate(results) if len(k_array) > 0]
    
    if not valid_indices:
        return np.empty((0, 3), dtype=np.int64)

    # 2. Extract the data
    flat_indices = np.array([item[0] for item in valid_indices])
    k_arrays = [item[1] for item in valid_indices]
    
    # 3. Calculate how many 'k's exist for each (i, j) pair
    counts = np.array([len(k) for k in k_arrays])
    
    # 4. Expand i and j
    # i = flat_idx // num, j = flat_idx % num
    i_vals = (flat_indices // num).astype(np.int64)
    j_vals = (flat_indices % num).astype(np.int64)
    
    # Repeat i and j based on how many k-values they have
    i_col = np.repeat(i_vals, counts)
    j_col = np.repeat(j_vals, counts)
    
    # 5. Concatenate all k arrays into one long column
    k_col = np.concatenate(k_arrays)
    
    # 6. Stack them horizontally to get the (N, 3) shape
    triplets = np.column_stack((i_col, j_col, k_col))
    
    return triplets

# Usage:
# triplets = convert_to_triplets(result, len(P))
# Run
results = convert_to_triplets(counterexample_frequency_parallel(P), len(P))

Total triples: 225088567704
Counterexamples: 118473339
Counterexample percentage: 0.05263409874987384


In [5]:
import itertools

def check_triplet_ordering(i_val, j_val, k_val):
    """
    Checks pairwise preferences and confirms if 'i' is the head of the 
    globally optimal RMSE ordering.
    """
    
    # Store the tuples in a dictionary for easy access by key
    data = {'i': i_val, 'j': j_val, 'k': k_val}

    def get_rmse_sq(order):
        elapsed = 0
        total_score = 0
        for x in order:
            N, T, W = data[x]
            elapsed += T
            total_score += N * ((W + elapsed) ** 2)
        return total_score # / total_n if (total_n := total_N) > 0 else 0

    # 1. Pairwise Assertions (A ≺ B means RMSE(A,B) < RMSE(B,A))
    # Check i ≺ j
    assert get_rmse_sq(('i', 'j')) <= get_rmse_sq(('j', 'i')), "Failed: i ≺ j"
    
    # Check j ≺ k
    assert get_rmse_sq(('j', 'k')) <= get_rmse_sq(('k', 'j')), ("Failed: j ≺ k", get_rmse_sq(('j', 'k')) - get_rmse_sq(('k', 'j'))) 
    
    # Check k ≺ i
    assert get_rmse_sq(('k', 'i')) <= get_rmse_sq(('i', 'k')), "Failed: k ≺ i"

    # 2. Global Evaluation
    perms = list(itertools.permutations(['i', 'j', 'k']))
    # Find the permutation with the minimum RMSE squared
    best_perm = min(perms, key=get_rmse_sq)
    
    # 3. Final Check: Is the first element of the best global order 'i'?
    is_i_first = (best_perm[0] == 'i')
    
    return is_i_first, best_perm

# Example Usage:
# N T W
i = (1, 1, 1)
j = (2, 4, 3)
k = (1, 3, 7)

try:
    success, best_order = check_triplet_ordering(i, j, k)
    print(f"Is 'i' first in global best? {success}")
    print(f"Best global order: {best_order}")
except AssertionError as e:
    print(f"Assertion failed: {e}")

Is 'i' first in global best? True
Best global order: ('i', 'j', 'k')


In [6]:

@njit(fastmath=True, nogil=True, boundscheck=False)
def check_triplet_ordering(i_val, j_val, k_val):
    # unpack tuples
    Ni, Ti, Wi = i_val
    Nj, Tj, Wj = j_val
    Nk, Tk, Wk = k_val

    # helper: compute rmse_sq for a given order encoded as ints
    # 0=i, 1=j, 2=k
    def get_rmse_sq(a, b, c, Ni, Ti, Wi, Nj, Tj, Wj, Nk, Tk, Wk):
        elapsed = 0.0
        total = 0.0

        # first
        if a == 0:
            elapsed += Ti
            total += Ni * ((Wi + elapsed) ** 2)
        elif a == 1:
            elapsed += Tj
            total += Nj * ((Wj + elapsed) ** 2)
        else:
            elapsed += Tk
            total += Nk * ((Wk + elapsed) ** 2)

        # second
        if b == 0:
            elapsed += Ti
            total += Ni * ((Wi + elapsed) ** 2)
        elif b == 1:
            elapsed += Tj
            total += Nj * ((Wj + elapsed) ** 2)
        else:
            elapsed += Tk
            total += Nk * ((Wk + elapsed) ** 2)

        # third
        if c == 0:
            elapsed += Ti
            total += Ni * ((Wi + elapsed) ** 2)
        elif c == 1:
            elapsed += Tj
            total += Nj * ((Wj + elapsed) ** 2)
        else:
            elapsed += Tk
            total += Nk * ((Wk + elapsed) ** 2)

        return total

    # all permutations of (0,1,2)
    perms = (
        (0,1,2),
        (0,2,1),
        (1,0,2),
        (1,2,0),
        (2,0,1),
        (2,1,0),
    )

    best_val = 1e308
    best_first = -1

    for p in perms:
        val = get_rmse_sq(p[0], p[1], p[2], Ni, Ti, Wi, Nj, Tj, Wj, Nk, Tk, Wk)
        if val < best_val:
            best_val = val
            best_first = p[0]

    # return only is_i_first
    return best_first == 0

In [7]:
messups = []
for idx, (i, j, k) in enumerate(results):
    i_first = check_triplet_ordering(*map(idx_to_ntw.get, (i, j, k)))
    if not i_first:
        messups.append(((i, j, k)))#, best_order))
        # print(i, j, k)

In [8]:
len(results), len(messups)

(118473339, 78666990)

In [9]:
13502378/118473339

0.1139697598967815

In [ ]:
# Find rules in messups
@njit(fastmath=True, nogil=True, boundscheck=False)
def ascending_N(args):
    i, j, k = args
    return N[i] <= N[j] and N[j] <= N[k]
@njit(fastmath=True, nogil=True, boundscheck=False)
def decending_N(args):
    i, j, k = args
    return N[i] >= N[j] and N[j] >= N[k]

@njit(fastmath=True, nogil=True, boundscheck=False)
def ascending_W(args):
    i, j, k = args
    return W[i] <= W[j] and W[j] <= W[k]

@njit(fastmath=True, nogil=True, boundscheck=False)
def decending_W(args):
    i, j, k = args
    return W[i] >= W[j] and W[j] >= W[k]

@njit(fastmath=True, nogil=True, boundscheck=False)
def ascending_T(args):
    i, j, k = args
    return T[i] <= T[j] and T[j] <= T[k]

@njit(fastmath=True, nogil=True, boundscheck=False)
def decending_T(args):
    i, j, k = args
    return T[i] >= T[j] and T[j] >= T[k]



@njit(fastmath=True, nogil=True, boundscheck=False)
def ascending_speed(args):
    i, j, k = args
    return N[i]/T[i] <= N[j]/T[j] and N[j]/T[j] <= N[k]/T[k]

# @njit(fastmath=True, nogil=True, boundscheck=False)
# def decending_speed(args):
#     i, j, k = args
#     return N[i]/T[i] >= N[j]/T[j] and N[j]/T[j] >= N[k]/T[k]

@njit(fastmath=True, nogil=True, boundscheck=False)
def decending_T_and_W_sum(args):
    i, j, k = args
    return (T[i]+W[i]) >= (T[j]+W[j]) and (T[j]+W[j]) >= (T[k]+W[k])


possible_rules = [ascending_N, decending_N, ascending_W, decending_W, ascending_T, decending_T, ascending_speed, decending_T_and_W_sum]

for rule in possible_rules:
    print(rule.__name__, sum(map(rule, messups)))

ascending_N 24693868
decending_N 14962716
ascending_W 242494
decending_W 31600964
ascending_T 7493657
decending_T 34794057
ascending_speed 29279051
decending_speed 287355
decending_T_and_W_sum 32238198


In [13]:

@njit(fastmath=True, nogil=True, boundscheck=False)
def decending_T_and_W_sum_half(args):
    i, j, k = args
    return (T[i]/2+W[i]) >= (T[j]/2+W[j]) and (T[j]/2+W[j]) >= (T[k]/2+W[k])

possible_rules = [decending_T_and_W_sum_half]

for rule in possible_rules:
    print(rule.__name__, sum(map(rule, messups)))

decending_T_and_W_sum_half 31136692
